# Signal Choice on a Crypto Pair

This notebook compares the three signal constructions discussed in Section 3.1 using local crypto fixtures:

- a dynamic price spread with fixed-share interpretation;
- a dynamic log-price spread with fixed-capital interpretation;
- a simple ratio.

The point is not to declare one universal winner. The point is to make the portfolio interpretation visible before any trading rule is attached.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.float_format", lambda x: f"{x:,.6f}")


def find_repo_root(start: Path | None = None) -> Path:
    current = Path.cwd() if start is None else start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / "fixtures/crypto/crypto_daily_close.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Missing shared crypto fixture: fixtures/crypto/crypto_daily_close.csv. "
        "Run `python3 scripts/python/download-crypto-fixtures.py --source binance-monthly-archive` "
        "from the repository root."
    )


repo_root = find_repo_root()
close = pd.read_csv(
    repo_root / "fixtures/crypto/crypto_daily_close.csv",
    parse_dates=["date"],
).set_index("date").sort_index()
close = close.apply(pd.to_numeric, errors="coerce").dropna(how="all")

symbols = ["BTCUSDT", "ETHUSDT"]
missing = [symbol for symbol in symbols if symbol not in close]
if missing:
    raise ValueError(f"Fixture is missing required symbols: {missing}")

prices = close[symbols].dropna()
prices.tail()


In [ ]:
def rolling_hedge_ratio(y: pd.Series, x: pd.Series, lookback: int) -> pd.Series:
    beta = pd.Series(np.nan, index=y.index, name="hedge_ratio")
    for t in range(lookback - 1, len(y)):
        y_window = y.iloc[t - lookback + 1 : t + 1].to_numpy()
        x_window = x.iloc[t - lookback + 1 : t + 1].to_numpy()
        design = np.column_stack([x_window, np.ones(lookback)])
        beta.iloc[t] = np.linalg.lstsq(design, y_window, rcond=None)[0][0]
    return beta


def zscore(series: pd.Series, lookback: int) -> pd.Series:
    mean = series.rolling(lookback).mean()
    std = series.rolling(lookback).std()
    return (series - mean) / std


def strategy_returns(prices: pd.DataFrame, hedge_ratio: pd.Series, units: pd.Series) -> pd.Series:
    aligned = pd.concat([prices, hedge_ratio.rename("hedge_ratio"), units.rename("units")], axis=1).dropna()
    btc = aligned["BTCUSDT"]
    eth = aligned["ETHUSDT"]
    shares = pd.DataFrame(
        {
            "BTCUSDT": -aligned["hedge_ratio"] * aligned["units"],
            "ETHUSDT": aligned["units"],
        },
        index=aligned.index,
    )
    dollar_positions = shares.mul(aligned[["BTCUSDT", "ETHUSDT"]])
    pnl = (dollar_positions.shift(1) * aligned[["BTCUSDT", "ETHUSDT"]].pct_change()).sum(axis=1)
    gross = dollar_positions.shift(1).abs().sum(axis=1)
    return (pnl / gross).replace([np.inf, -np.inf], np.nan).dropna()


def performance_stats(returns: pd.Series) -> pd.Series:
    returns = returns.dropna()
    if returns.empty or returns.std() == 0:
        return pd.Series({"days": len(returns), "APR": np.nan, "Sharpe": np.nan, "max_drawdown": np.nan})
    equity = (1 + returns).cumprod()
    drawdown = equity / equity.cummax() - 1
    return pd.Series(
        {
            "days": len(returns),
            "APR": equity.iloc[-1] ** (252 / len(returns)) - 1,
            "Sharpe": np.sqrt(252) * returns.mean() / returns.std(),
            "max_drawdown": drawdown.min(),
        }
    )

from statsmodels.tsa.stattools import adfuller


def adf_summary(series: pd.Series, name: str) -> dict[str, float | str | bool]:
    clean = series.dropna()
    stat, p_value, used_lag, n_obs, critical, _ = adfuller(clean, regression="c", autolag="AIC")
    return {
        "signal": name,
        "adf_stat": stat,
        "p_value": p_value,
        "used_lag": used_lag,
        "n_obs": n_obs,
        "crit_5pct": critical["5%"],
        "reject_5pct": bool(stat < critical["5%"]),
    }


In [ ]:
lookback = 20
x = prices["BTCUSDT"]
y = prices["ETHUSDT"]

price_beta = rolling_hedge_ratio(y, x, lookback)
price_spread = (y - price_beta * x).rename("dynamic price spread")

log_prices = np.log(prices)
log_beta = rolling_hedge_ratio(log_prices["ETHUSDT"], log_prices["BTCUSDT"], lookback)
log_spread = (log_prices["ETHUSDT"] - log_beta * log_prices["BTCUSDT"]).rename("dynamic log spread")

ratio = (y / x).rename("ratio")

signals = pd.concat([price_spread, log_spread, ratio], axis=1).dropna()
signals.tail()


In [ ]:
standardized = (signals - signals.mean()) / signals.std()

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for ax, column in zip(axes, standardized.columns):
    standardized[column].plot(ax=ax, linewidth=2)
    ax.axhline(0, color="black", linewidth=1)
    ax.set_title(column)
    ax.set_ylabel("standardized level")
plt.tight_layout();


In [ ]:
adf_results = pd.DataFrame(adf_summary(signals[column], column) for column in signals)
adf_results.set_index("signal")


In [ ]:
price_units = -zscore(price_spread, lookback).rename("units")
price_returns = strategy_returns(prices, price_beta, price_units)

log_signal_units = -zscore(log_spread, lookback)
ratio_units = -zscore(ratio, lookback)

summary = pd.DataFrame(
    {
        "dynamic price spread": performance_stats(price_returns),
        "log spread signal z-score": performance_stats(log_signal_units.shift(1).dropna() * log_prices["ETHUSDT"].diff().dropna()),
        "ratio signal z-score": performance_stats(ratio_units.shift(1).dropna() * ratio.pct_change().dropna()),
    }
).T
summary


## Interpretation

A price spread maps naturally to fixed shares during the trade. A log spread maps naturally to capital weights and therefore implies rebalancing. A ratio can be useful when proportional movement is the economic relation, but it is only a special cointegration case when the normalized hedge ratios have equal magnitude and opposite sign.
